In [1]:
### --- Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import keras
import tensorflow as tf

### --- random state setting
seed=42

I0000 00:00:1779045432.241298  339856 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1779045432.906538  339856 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779045440.171369  339856 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


# Preprocesamiento

In [2]:
from pathlib import Path
from glob import glob

base_path = Path("synthetic_images")
#imgs_path = base_path / "images"
reg_path = base_path / "regression.csv"

df_reg = pd.read_csv(reg_path, sep=",")
df_reg = df_reg.rename({"values" : "b1"}, axis=1)
df_reg.head()

,images,b1,halo_mass,d3,d5,d8,c_200,z,spin,d_min,d_node,d_saddle_1,d_saddle_2,d_skel
0,images/000000.png,-0.217024,12.071234,3.740577,1.976566,0.993058,11.072979,0.545455,0.095848,32859.226901,1902.399980,7193.204792,6661.655357,265.254706
1,images/000001.png,1.437520,11.723960,1.765938,1.459885,1.245344,9.400798,1.696970,0.077347,39457.337930,9206.159552,2083.272550,1691.418306,620.803580
2,images/000002.png,-0.071316,11.663453,1.814100,1.154731,0.897392,11.708395,1.575758,0.067337,42913.485948,10387.286207,8094.810187,9349.823871,892.387973
3,images/000003.png,4.479510,11.323860,1.123779,1.182472,2.851337,8.740814,1.272727,0.062813,24593.767358,5616.238855,5052.636900,5802.865947,5124.807584
4,images/000004.png,0.509487,12.148274,4.896464,2.455103,1.567050,11.504637,1.939394,0.052333,21648.720090,1621.941740,9256.419752,488.090764,535.375829


In [3]:
from keras.applications import ResNet50V2
from keras.layers import GlobalAveragePooling2D, Dense, Dropout
from keras.models import Model

def make_model():
  base_model = ResNet50V2(
      weights="imagenet",
      input_shape = (224, 224,3),
      include_top=False
  )

  base_model.trainable = False

  # Construir la cabeza de regresión
  inputs = base_model.input
  x = base_model.output

  # Pooling global para reducir dimensiones
  x = GlobalAveragePooling2D()(x)

  # Capas densas para regresión
  x = Dense(512, activation='relu')(x)
  x = Dropout(0.5)(x)  # Regularización para evitar overfitting
  x = Dense(512, activation='relu')(x)
  x = Dropout(0.3)(x)
  x = Dense(512, activation='relu')(x)

  # Capa de salida: 1 neurona, sin activación (regresión lineal)
  outputs = Dense(1, activation='linear')(x)

  # Crear modelo
  model = Model(inputs=inputs, outputs=outputs)

  return model

In [5]:
from keras.optimizers import Adam

model = make_model()
lr = 1e-4
model.compile(
    optimizer=Adam(learning_rate=lr),
    loss='mse',  # Mean Squared Error para regresión
    metrics=['mae', 'mse']  # Mean Absolute Error también es útil
)

#model.summary()

In [10]:
from sklearn.model_selection import train_test_split

features = ['images','halo_mass', 'd3', 'd5', 'd8', 'c_200', 'z', 'spin', 'd_min', 'd_node',
       'd_saddle_1', 'd_saddle_2', 'd_skel']
target = ['b1']

X = df_reg[features]
y = df_reg[target]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.20, random_state = 123)
X_val, X_test, y_val, y_test = train_test_split(X_val, y_val, test_size = 0.20, random_state = 123)
df_train = pd.concat([X_train, y_train], axis = 1)
df_test = pd.concat([X_test, y_test], axis = 1)
df_val = pd.concat([X_val, y_val], axis = 1)

In [12]:
from keras_preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(rescale = 1./255)
valid_datagen = ImageDataGenerator(rescale = 1./255)
test_datagen = ImageDataGenerator(rescale = 1./255)

train_iter = train_datagen.flow_from_dataframe(
    df_train,
    target_size = (224, 224),
    directory=base_path,
    x_col = 'images',
    y_col = 'b1',
    class_mode = 'raw',
    batch_size = 8,
    shuffle = True
)

valid_iter = train_datagen.flow_from_dataframe(
    df_val,
    target_size = (224, 224),
    directory=base_path,
    x_col = 'images',
    y_col = 'b1',
    class_mode = 'raw',
    batch_size = 8
)

test_iter = train_datagen.flow_from_dataframe(
    df_test,
    target_size = (224, 224),
    directory=base_path,
    x_col = 'images',
    y_col = 'b1',
    class_mode = 'raw',
    batch_size = 8,
    shuffle = True
)

Found 11733 validated image filenames.
Found 2347 validated image filenames.
Found 587 validated image filenames.


In [13]:
train_iter.image_shape

(224, 224, 3)

In [ ]:
df.dtypes

In [ ]:
df.columns

In [ ]:
### --- Normalize data
from sklearn.preprocessing import StandardScaler

features = ['halo_mass', 'd3', 'd5', 'd8', 'c_200', 'z', 'spin', 'd_min', 'd_node',
       'd_saddle_1', 'd_saddle_2', 'd_skel']
target = ['b1']

X = df[features]
y = df[target]


In [ ]:
sns.heatmap(X.corr())

In [ ]:
scaler = StandardScaler()
X_norm = scaler.fit_transform(X)

scaler = StandardScaler()
y_norm = scaler.fit_transform(y)

X_norm.shape, y_norm.shape

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=0.95)
fit = pca.fit(X_norm)
X_norm = fit.transform(X_norm)
X_norm.shape, fit.explained_variance_

In [ ]:
### --- train-val-test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_norm,y_norm,test_size=0.2,random_state=seed)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=seed)
X_train.shape, X_val.shape, X_test.shape

# Estimando bias usando modelo ResNet50 (con convoluciones 1D)

In [ ]:
os.environ["KERAS_BACKEND"] = "tensorflow"
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
tf.test.gpu_device_name()

In [ ]:
tf.config.list_physical_devices()

## Arquitectura (advertencia: es largo)

In [ ]:
### --- ResNet50
### --- https://github.com/Jun-depo/Resnet50_Conv1D/blob/master/Resnet50Conv1d_BN.py#L126

import tensorflow as tf

import keras
import keras.backend as K

from keras.utils import plot_model, to_categorical
from keras.models import Sequential, load_model
from keras.layers import Input,Activation, Dropout, Flatten, Dense, Conv1D, BatchNormalization, LSTM
from keras import regularizers

from keras.callbacks import ModelCheckpoint
from keras import metrics
from keras.optimizers import Adam

from keras.models import *
from keras.layers import *
from keras.callbacks import *
from keras.initializers import *
from keras.layers import Layer

def identity_block(X, f, filter_numbers, stage, block):
    """
    Implementation of the identity block as defined in Figure 4.

    Arguments:
    X -- input tensor of shape (m, input_length_prev (n_w), input_Channel_prev (n_c).
    f -- kernel_size, integer, shape of convolution filter in the main path.
    filter_numbers -- python list of integers, defining the number of filters in the CONV layers of the main path.
    stage -- integer, used to name the layers, depending on their position in the network.
    block -- string/character, used to name the layers, depending on their position in the network.

    Returns:
    X -- output of the identity block, tensor of shape (n_W, n_C)
    """

    # defining name basis
    conv_name_base = 'res' + str(stage) + block + '_branch'
    bn_name_base = 'bn' + str(stage) + block + '_branch'

    # Retrieve Filters
    F1, F2, F3 = filter_numbers

    # Save the input value. You'll need this later to add back to the main path.
    X_shortcut = X

    # First Conv layer
    X = Conv1D(filters = F1, kernel_size = 1, strides = 1, padding = 'valid', name = conv_name_base + '2a', kernel_initializer = glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis = 2, name = bn_name_base + '2a')(X)
    X = Activation('relu')(X)



    # Second Conv layer
    X = Conv1D(filters = F2, kernel_size = f, strides = 1, padding = 'same', name = conv_name_base + '2b', kernel_initializer = glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis = 2, name = bn_name_base + '2b')(X)
    X = Activation('relu')(X)

    # Third Conv layer
    X = Conv1D(filters = F3, kernel_size = 1, strides = 1, padding = 'valid', name = conv_name_base + '2c', kernel_initializer = glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis = 2, name = bn_name_base + '2c')(X)

    # Merge with Residual shortcut
    X = Add()([X, X_shortcut])
    X = Activation('relu')(X)

    ### END CODE HERE ###

    return X

def convolutional_block(X, f, filters, stage, block, s = 2):
    """
    Implementation of the convolutional block as defined in Figure 4

    Arguments:
    X -- input tensor of shape (m, input_length_prev (n_w), input_Channel_prev (n_c))
    f -- integer, specifying the shape of the middle CONV's window for the main path
    filters -- python list of integers, defining the number of filters in the CONV layers of the main path
    stage -- integer, used to name the layers, depending on their position in the network
    block -- string/character, used to name the layers, depending on their position in the network
    s -- Integer, specifying the stride to be used

    Returns:
    X -- output of the convolutional block, tensor of shape (n_W, n_C)
    """

    # defining name basis
    conv_name_base = 'res' + str(stage) + block + '_branch'
    bn_name_base = 'bn' + str(stage) + block + '_branch'

    # Retrieve Filters
    F1, F2, F3 = filters

    # Save the input value
    X_shortcut = X


    ##### MAIN PATH #####
    # First component of main path
    X = Conv1D(F1, 1, strides = s, name = conv_name_base + '2a', kernel_initializer = glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis = 2, name = bn_name_base + '2a')(X)
    X = Activation('relu')(X)

    ### START CODE HERE ###

    # Second component of main path (≈3 lines)
    X = Conv1D(F2, f, strides = 1, padding = "same", name = conv_name_base + '2b', kernel_initializer = glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis = 2, name = bn_name_base + '2b')(X)
    X = Activation('relu')(X)

    # Third component of main path (≈2 lines)
    X = Conv1D(F3, 1, strides = 1, name = conv_name_base + '2c', kernel_initializer = glorot_uniform(seed=0))(X)
    #X = BatchNormalization(axis = 2, name = bn_name_base + '2c')(X)

    ##### SHORTCUT PATH #### (≈2 lines)
    X_shortcut = Conv1D(F3, 1, strides = s, name = conv_name_base + '1', kernel_initializer = glorot_uniform(seed=0))(X_shortcut)
    #X_shortcut = BatchNormalization(axis = 2, name = bn_name_base + '1')(X_shortcut)

    # Final step: Add shortcut value to main path, and pass it through a RELU activation (≈2 lines)
    X = Add()([X, X_shortcut])
    X = Activation('relu')(X)

    ### END CODE HERE ###

    return X

def ResNet50(input_shape = (30000, 1), max_pool_s=10, max_strides=5, kernel_size=3, strides = 2, f=3, ave_pool_size=5, n_out=1, i=None):
    """
    Implementation of the popular ResNet50 the following architecture:
    CONV1D -> BATCHNORM -> RELU -> MAXPOOL -> CONVBLOCK -> IDBLOCK*2 -> CONVBLOCK -> IDBLOCK*3
    -> CONVBLOCK -> IDBLOCK*5 -> CONVBLOCK -> IDBLOCK*2 -> AVGPOOL -> TOPLAYER

    Arguments:
    input_shape -- shape of the 1D data
    n_out -- integer, number of classes or output

    Returns:
    model -- a Model() instance in Keras

    params here were used in one of my projects.
    """

    # Define the input as a tensor with shape input_shape
    X_input = Input(input_shape)
    X = MaxPooling1D(max_pool_s, max_strides)(X_input)

    # Zero-Padding
    X = ZeroPadding1D(3)(X)

    # stage 1, 64 filters, kernel_size=7
    X = Conv1D(64, kernel_size, strides=1, name = 'conv1', kernel_initializer = glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis = 2, name = 'bn_conv1')(X)
    X = Activation('relu')(X)
    X = MaxPooling1D(3, strides=2)(X)

    # Stage 2
    X = convolutional_block(X, f, filters = [16, 16, 64], stage = 2, block='a', s = 1)
    X = identity_block(X, f, [16, 16, 64], stage=2, block='b')
    X = identity_block(X, f, [16, 16, 64], stage=2, block='c')

    ### START CODE HERE ###

    X = convolutional_block(X, f, filters = [32,32,128], stage = 3, block='a', s = 2)
    X = identity_block(X, f, [32,32,128], stage=3, block='b')
    X = identity_block(X, f, [32,32,128], stage=3, block='c')
    X = identity_block(X, f, [32,32,128], stage=3, block='d')

    # Stage 4 (≈6 lines)

    X = convolutional_block(X, f, filters = [64, 64, 256], stage = 4, block='a', s = 2)
    X = identity_block(X, f, [64, 64, 256], stage=4, block='b')
    X = identity_block(X, f, [64, 64, 256], stage=4, block='c')
    X = identity_block(X, f, [64, 64, 256], stage=4, block='d')
    X = identity_block(X, f, [64, 64, 256], stage=4, block='e')
    X = identity_block(X, f, [64, 64, 256], stage=4, block='f')

    # Stage 5

    X = convolutional_block(X, f, filters = [64, 64, 256], stage = 5, block='a', s = 2)
    X = identity_block(X, f, [64, 64, 256], stage=5, block='b')
    X = identity_block(X, f, [64, 64, 256], stage=5, block='c')

    X = AveragePooling1D(ave_pool_size)(X)

    # lstm layers
    # X = LSTM(units=1024, name='lstm-1',
    #          kernel_regularizer=regularizers.l2(0.2), bias_regularizer=regularizers.l2(0.2), return_sequences=True)(X)

    # X = Dropout(0.5)(X)

    # X = LSTM(units=256, name='lstm-2',
    #          kernel_regularizer=regularizers.l2(0.2), bias_regularizer=regularizers.l2(0.2))(X)

    # X = Dense(64, name='fc-dense-1', kernel_initializer=glorot_uniform(seed=0),
    #           kernel_regularizer=regularizers.l2(0.2), bias_regularizer=regularizers.l2(0.2))(X)

    X = Dropout(0.5)(X)

    # For regression
    X = Dense(n_out, name='fc-dense-2', kernel_initializer = glorot_uniform(seed=0),
              kernel_regularizer=regularizers.l2(0.2), bias_regularizer=regularizers.l2(0.2))(X)

    # for classification, if n_out =1, add:
    # X = Activation('sigmoid')(X)

    # for classification, if n_out > 1, add:
    # X = Activation('softmax')(X)

    # Create model
    model = Model(inputs = X_input, outputs = X, name=f'ResNet50_1d_{i}')

    return model

## Entrenamiento

In [ ]:
metrics = [
    'root_mean_squared_error',
    'mean_absolute_error',
    'mean_absolute_percentage_error',
]

lr = 1e-4
optim = Adam(learning_rate=lr)

with tf.device('/device:GPU:0'):
  models = []
  input = keras.layers.Input(shape=(X_train.shape[1],1))
  model = ResNet50(
        input_shape=(X_train.shape[1],1),
        max_pool_s=1,
        strides=1,
        f=2,
        max_strides=1,
        ave_pool_size=1
    )
  model.compile(
      loss = "mse",
      optimizer = optim,
      metrics = metrics
  )


In [ ]:
!touch Resnet50Conv1d.keras

In [ ]:
batch_size=64
epochs=1000

checkpointer = ModelCheckpoint(
    './Resnet50Conv1d.keras',
    verbose=0,
    save_best_only=True
)

history = model.fit(
    X_train,
    y_train,
    batch_size=batch_size,
    epochs=epochs,
    verbose=0,
    callbacks=[checkpointer],
    validation_data=(X_val, y_val)
)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15,12))
ax.set_title('loss')
ax.plot(history.epoch[20:], history.history["loss"][20:], label="Train loss")
ax.plot(history.epoch[20:], history.history["val_loss"][20:], label="Validation loss")
ax.legend()
plt.show()

In [ ]:
model.evaluate(X_test, y_test)